In [1]:
5

5

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Integrated\AppData\Local\Temp\ipykernel_11916\4211815356.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from langchain_community.vectorstores import Chroma

persist_directory = "persist_directory3"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

print("Vectorstore loaded successfully.")

C:\Users\Integrated\AppData\Local\Temp\ipykernel_11916\896937473.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Vectorstore loaded successfully.


In [5]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Get full docs once
store_data = vectorstore.get()
all_texts = store_data["documents"]
all_metadatas = store_data.get("metadatas", [{}] * len(all_texts))

# Build Document objects with metadata
documents = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(all_texts, all_metadatas)
]

# Dense retriever (larger k for candidate pool)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# BM25 retriever
bm25 = BM25Retriever.from_documents(documents)
bm25.k = 20


def hybrid_retrieve(query, k_dense=20, k_bm25=20, max_chunks=30):
    
    # Update k dynamically if needed
    dense_retriever.search_kwargs["k"] = k_dense
    bm25.k = k_bm25
    
    # Dense retrieval
    dense_docs = dense_retriever.invoke(query)
    
    # Sparse retrieval
    bm25_docs = bm25.invoke(query)
    
    # Combine
    combined = dense_docs + bm25_docs
    
    # Deduplicate by content
    seen = set()
    unique_docs = []
    
    for doc in combined:
        if doc.page_content not in seen:
            unique_docs.append(doc)
            seen.add(doc.page_content)
    
    return unique_docs[:max_chunks]

In [6]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_rerank(query, top_k_final=5):
    
    candidate_docs = hybrid_retrieve(query)
    
    pairs = [(query, doc.page_content) for doc in candidate_docs]
    scores = reranker.predict(pairs)
    
    reranked = sorted(
        zip(candidate_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    return [doc for doc, score in reranked[:top_k_final]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

d:\Adrta\rag_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [7]:
template = """
            You are a helpful AI research assistant.

            Use ONLY the provided context to answer the question.
            If the answer is not explicitly present in the context, say:
            "I don't know based on the provided context."

            Some Authors of the paper can be from - Indus University, Ahmedabad, India


            Give the answer in detail, search if necessary.

            Context:
            {context}

            Question:
            {question}

            Answer:
            """

In [15]:
query = "What is the abstract in the paper?"
results = hybrid_rerank(query, top_k_final=5)

In [17]:
context = "\n\n".join([doc.page_content for doc in results])

final_prompt = template.format(
    context=context,
    question=query
)

response = client2.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=final_prompt,
)

print("\nAnswer:\n")
print(response.text)


Answer:

This paper presents a data-driven approach for predicting the stability of model rockets using both aerodynamic and environmental features. Due to the scarcity and proprietary nature of actual flight data, the authors generate realistic synthetic data using OpenRocket simulations and data augmentation techniques. Machine learning models, specifically Random Forest and Neural Networks, are then trained to predict rocket stability. The core contribution of this work is a research-driven, replicable methodology for using simulated data and machine learning in a field where empirical experimentation is limited by safety, cost, and scale. This opens up new avenues for data-centric aerospace modeling and integrates AI tools into model rocket design, particularly in educational and exploratory contexts.


In [13]:
query = "Explain the augmentation methods used in the paper in detail."
results = hybrid_rerank(query, top_k_final=5)

In [14]:
context = "\n\n".join([doc.page_content for doc in results])

final_prompt = template.format(
    context=context,
    question=query
)

response = client2.models.generate_content(
    model="gemini-2.5-flash",
    contents=final_prompt,
)

print("\nAnswer:\n")
print(response.text)


Answer:

The paper utilized two main augmentation techniques to improve data diversity, model robustness, and address class imbalance:

1.  **Gaussian Noise Augmentation**:
    *   **Purpose**: This technique was applied to simulate real-world sensor inaccuracies and environmental fluctuations, thereby enhancing data diversity and model robustness.
    *   **Methodology**: Gaussian noise was injected into specific selected continuous features of the dataset.
    *   **Affected Features**: The continuous features augmented with Gaussian noise included `thrust_N`, `rocket_length_cm`, `wind_speed_mps`, and `center_of_mass_cm`.
    *   **Mathematical Representation**: The augmentation was performed using the formula: `𝒙aug = 𝒙+ 𝝐`, where `𝒙aug` is the augmented value, `𝒙` is the original value, and `𝝐` represents Gaussian noise. This noise `𝝐` follows a normal distribution `𝓝(𝟎, 𝝈𝟐)` with a mean of zero and a variance `σ2`, which was typically kept small to ensure realistic perturbations.

In [23]:
def rag_pipeline(query):
    
    top_docs = hybrid_rerank(query, top_k_final=5)
    context = "\n\n".join([doc.page_content for doc in top_docs])
    prompt = template.format(context=context, question=query)
    answer = client2.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    ).text
    
    sources = [doc.metadata for doc in top_docs]
    
    return {
        "answer": answer
    }

In [24]:
query = "What is the title of the paper?"
print(rag_pipeline(query))

{'answer': 'Stability Before Liftoff: Predictive Analytics for Small Rocket Design with Hybrid Synthetic Aerodynamic & Environmental Data'}
